In [0]:
# Databricks notebook source
import os
import yaml
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
caminho_config = os.path.join(repo_root, "config", "series_sgs.yaml")
with open(caminho_config, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

catalogo = config["catalogo"]
tabela_bronze = f"{catalogo}.bronze.sgs_raw"
tabela_silver = f"{catalogo}.silver.sgs_series_tratadas"

# COMMAND ----------

# 1. Leitura da Bronze, tipagem e deduplicação
df_raw = (
    spark.read.table(tabela_bronze)
    .filter(F.col("data_particao") >= F.date_sub(F.current_date(), 3))
)

df_tratado = (
    df_raw
    .withColumn("data_referencia", F.to_date(F.col("data"), "dd/MM/yyyy"))
    .withColumn("valor_numerico", F.col("valor").cast("decimal(18,6)"))
    .filter(F.col("data_referencia").isNotNull() & F.col("valor_numerico").isNotNull())
)

# Deduplicação mantendo o registro mais recente ingerido para cada (codigo_serie, data_referencia)
janela_dedup = Window.partitionBy("codigo_serie", "data_referencia").orderBy(F.col("timestamp_ingestao").desc())

df_silver_stage = (
    df_tratado
    .withColumn("rn", F.row_number().over(janela_dedup))
    .filter(F.col("rn") == 1)
    .select(
        F.col("codigo_serie").alias("id_serie"),
        F.col("nome_serie"),
        F.col("periodicidade"),
        F.col("unidade").alias("unidade_medida"),
        F.col("data_referencia"),
        F.col("valor_numerico").alias("valor"),
        F.current_timestamp().alias("data_atualizacao")
    )
)

# COMMAND ----------

# 2. Carga Incremental (Upsert / MERGE INTO) na camada Silver
if not spark.catalog.tableExists(tabela_silver):
    (
        df_silver_stage.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_silver)
    )
else:
    delta_silver = DeltaTable.forName(spark, tabela_silver)
    (
        delta_silver.alias("target")
        .merge(
            df_silver_stage.alias("source"),
            "target.id_serie = source.id_serie AND target.data_referencia = source.data_referencia"
        )
        .whenMatchedUpdate(
            condition="target.valor <> source.valor OR target.nome_serie <> source.nome_serie",
            set={
                "nome_serie": "source.nome_serie",
                "periodicidade": "source.periodicidade",
                "unidade_medida": "source.unidade_medida",
                "valor": "source.valor",
                "data_atualizacao": "source.data_atualizacao"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )